In [55]:
import yfinance as yf
import numpy as np
import pandas as pd

In [106]:
tickers = ["PLY.AX","LAU.AX","TLX.AX","COS.AX","ANG.AX",
           "VVA.AX","WTC.AX","AUB.AX","XYZ.AX","DUG.AX"]

industries = pd.Series({
    "PLY.AX": "Communication Services", 
    "LAU.AX": "Industrials", "ANG.AX": "Industrials",
    "TLX.AX": "Health Care", 
    "COS.AX": "Information Technology", "WTC.AX": "Information Technology", "DUG.AX": "Information Technology",
    "VVA.AX": "Consumer Discretionary",
    "AUB.AX": "Financials", "XYZ.AX": "Financials"
})

data = yf.download(tickers, start="2025-01-13", end="2025-12-31")
shares_outstanding = {t: yf.Ticker(t).info.get('sharesOutstanding') for t in tickers}
epsilon = 1e-10
prices = data["Close"]
returns = prices.pct_change().dropna()

[*********************100%***********************]  10 of 10 completed


In [107]:
data

Price          Close                                                          \
Ticker        ANG.AX     AUB.AX    COS.AX DUG.AX    LAU.AX PLY.AX     TLX.AX   
Date                                                                           
2025-01-13  0.504255  28.592453  0.930424  1.265  0.800259  0.410  24.020000   
2025-01-14  0.504255  29.179508  0.930424  1.300  0.795686  0.410  24.980000   
2025-01-15  0.504255  29.237251  0.906195  1.290  0.786540  0.410  25.110001   
2025-01-16  0.504255  29.227629  0.891657  1.295  0.804832  0.430  25.799999   
2025-01-17  0.504255  29.448977  0.891657  1.315  0.804832  0.440  26.590000   
...              ...        ...       ...    ...       ...    ...        ...   
2025-12-22  0.196923  30.263908  0.450000  2.120  0.675702  0.240  11.910000   
2025-12-23  0.206769  30.491306  0.465000  2.140  0.680529  0.260  11.940000   
2025-12-24  0.211692  30.204584  0.465000  2.100  0.656397  0.255  11.890000   
2025-12-29  0.226462  30.234247  0.475000  2.180  0.661223  0.255  11.770000   
2025-12-30  0.236308  30.629723  0.490000  2.120  0.670876  0.260  11.620000   

Price                                      ...   Volume                  \
Ticker     VVA.AX      WTC.AX      XYZ.AX  ...   ANG.AX  AUB.AX  COS.AX   
Date                                       ...                            
2025-01-13  1.365  120.476189  134.309998  ...   768522  180026   39304   
2025-01-14  1.350  118.863472  133.889999  ...   718441  207775   20580   
2025-01-15  1.350  114.483238  133.789993  ...   189907  394951  211309   
2025-01-16  1.335  117.469765  138.380005  ...   200060  138071   45828   
2025-01-17  1.375  116.543945  139.250000  ...   223521  115606       0   
...           ...         ...         ...  ...      ...     ...     ...   
2025-12-22  1.650   67.093964   99.300003  ...  2296927  263396    4640   
2025-12-23  1.615   68.600906   97.839996  ...   890637  168254  106957   
2025-12-24  1.615   68.630844   96.500000  ...   466741  103116      89   
2025-12-29  1.595   67.932266   98.680000  ...  2478460  132137   19967   
2025-12-30  1.620   67.682770   98.300003  ...  2951154  173877    6484   

Price                                                                
Ticker      DUG.AX  LAU.AX  PLY.AX   TLX.AX VVA.AX   WTC.AX  XYZ.AX  
Date                                                                 
2025-01-13  235123   96117  270483   473392   6318   334503  435103  
2025-01-14  504770  302181   82321  1180715  28298   302020  285377  
2025-01-15  387335  134878  283809   834776  12096   705056  327989  
2025-01-16  460156  240594  288839   867463  89931   622991  303273  
2025-01-17  343373  119964   86965  1210887  26993   532007  171388  
...            ...     ...     ...      ...    ...      ...     ...  
2025-12-22   31989  865410  702340  2124649  13661  4119529   48554  
2025-12-23   62198  416539  913887  1327430   6883   647643   65142  
2025-12-24   10692  360227   98600   770307      0   384206   28034  
2025-12-29  296360  227541  113901   983109  14278   505742   51433  
2025-12-30   29259  160032  254587  1663631  18639   472214   19295  

[245 rows x 50 columns]

In [108]:
# Momentum (12-1 style approx)
momentum = (prices.shift(21) / prices.shift(252)) - 1
#momentum = (prices.shift(30) / monthly_df.shift(252)) - 1


# Volatility (risk)
volatility = returns.rolling(60).std()

# Liquidity (log dollar volume)
dollar_volume = data["Volume"] * prices
liquidity = np.log(dollar_volume.rolling(20).mean())

# Size
size = np.log(prices.multiply(pd.Series(shares_outstanding)))  # proxy for market cap

In [133]:
momentum = momentum.shift(1)
volatility = volatility.shift(1)
liquidity = liquidity.shift(1)
size = size.shift(1)

In [134]:
common_dates = returns.index.intersection(exposure.index.get_level_values(0))

returns = returns.loc[common_dates]
exposure = exposure.loc[common_dates]

momentum = momentum.loc[common_index]
volatility = volatility.loc[common_index]
liquidity = liquidity.loc[common_index]
size = size.loc[common_index]

In [135]:
def winsorize(df, limit=3):
    return df.clip(-limit, limit)

momentum = winsorize(momentum)
volatility = winsorize(volatility)
liquidity = winsorize(liquidity)
size = winsorize(size)

In [136]:
def cs_zscore(df):
    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1), axis=0)

momentum_z = cs_zscore(momentum)
volatility_z = cs_zscore(volatility)
liquidity_z = cs_zscore(liquidity)
size_z = cs_zscore(size)

In [137]:
industry_map = {
    "PLY.AX": "Industrials",
    "LAU.AX": "Industrials",
    "TLX.AX": "Healthcare",
    "COS.AX": "Industrials",
    "ANG.AX": "Energy",
    "VVA.AX": "Financials",
    "WTC.AX": "Tech",
    "AUB.AX": "Financials",
    "XYZ.AX": "Materials",
    "DUG.AX": "Energy"
}

industry_df = pd.Series(industry_map)
industry_dummies = pd.get_dummies(industry_df).astype(float)

# drop one column to avoid multicollinearity
industry_dummies = industry_dummies.iloc[:, :-1]

# expand across time to match (Date, Ticker)
industry_expanded = pd.concat(
    [industry_dummies] * len(returns),
    keys=returns.index
)

industry_expanded.index.names = ["Date", "Ticker"]

In [144]:
factors = pd.concat({
    "momentum": momentum_z,
    "volatility": volatility_z,
    "liquidity": liquidity_z,
    "size": size_z
}, axis=1)

# reshape
exposure = factors.stack(level=1)
exposure.index.names = ["Date", "Ticker"]

# add industry
exposure = exposure.join(industry_expanded, how="left")
exposure = exposure.fillna(0)
# get valid dates from exposure
valid_dates = exposure.index.get_level_values(0).unique()

# intersect with returns
common_dates = returns.index.intersection(valid_dates)

returns = returns.loc[common_dates]
exposure = exposure.loc[common_dates]

/var/folders/hk/kwysyfbx16v885w2xpzqxp5m0000gn/T/ipykernel_30429/2974227637.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  exposure = factors.stack(level=1)


# Test 1 (Working I think)

In [145]:
# reshape exposure into dictionary of B_t matrices
B_dict = {}

for t in returns.index:
    B_t = exposure.loc[t]   # (m × K)
    B_dict[t] = B_t.values

In [146]:
factor_returns = []

for t in returns.index:
    x_t = returns.loc[t].values
    B_t = B_dict[t]

    # OLS
    f_t = np.linalg.pinv(B_t) @ x_t


    factor_returns.append(f_t)

factor_returns = pd.DataFrame(
    factor_returns, 
    index=returns.index, 
    columns=exposure.columns
)

In [147]:
residuals = []

for t in returns.index:
    x_t = returns.loc[t].values
    B_t = B_dict[t]
    f_t = factor_returns.loc[t].values

    eps_t = x_t - B_t @ f_t
    residuals.append(eps_t)

residuals = pd.DataFrame(residuals, index=returns.index, columns=returns.columns)

In [148]:
Omega_f = factor_returns.cov()

specific_var = residuals.var()
Psi = np.diag(specific_var)

In [149]:
B_latest = exposure.loc[returns.index[-1]].values  # (m × K)

Sigma = B_latest @ Omega_f.values @ B_latest.T + Psi
Sigma = pd.DataFrame(Sigma, index=returns.columns, columns=returns.columns)

In [150]:
# equal weights
w = np.ones(len(tickers)) / len(tickers)

portfolio_var_daily = w.T @ Sigma.values @ w
portfolio_vol_daily = np.sqrt(portfolio_var_daily)

trading_days = 252

portfolio_var_annual = portfolio_var_daily * trading_days
portfolio_vol_annual = portfolio_vol_daily * np.sqrt(trading_days)

print("Daily Vol:", portfolio_vol_daily)
print("Annual Vol:", portfolio_vol_annual)

Daily Vol: 0.013049385278333657
Annual Vol: 0.20715256925242945


In [151]:
Sigma_annual = Sigma * 252
print("Daily covariance matrix:")
print(Sigma.round(6))

print("\nAnnual covariance matrix:")
print(Sigma_annual.round(6))

Daily covariance matrix:
Ticker    ANG.AX    AUB.AX    COS.AX    DUG.AX    LAU.AX    PLY.AX    TLX.AX  \
Ticker                                                                         
ANG.AX  0.001651 -0.000050 -0.000069  0.000883  0.000046 -0.000132  0.000144   
AUB.AX -0.000050  0.000996  0.000452 -0.000702 -0.000491  0.000973 -0.000093   
COS.AX -0.000069  0.000452  0.001943 -0.000924 -0.000181  0.001737 -0.000227   
DUG.AX  0.000883 -0.000702 -0.000924  0.003016  0.000933 -0.001950  0.000466   
LAU.AX  0.000046 -0.000491 -0.000181  0.000933  0.001526 -0.000889  0.000238   
PLY.AX -0.000132  0.000973  0.001737 -0.001950 -0.000889  0.003650 -0.000483   
TLX.AX  0.000144 -0.000093 -0.000227  0.000466  0.000238 -0.000483  0.001427   
VVA.AX -0.000036  0.000565  0.000310 -0.000489 -0.000344  0.000672 -0.000040   
WTC.AX  0.000012 -0.000094 -0.000124  0.000186  0.000128 -0.000263  0.000047   
XYZ.AX  0.000192 -0.000189 -0.000381  0.000487  0.000046 -0.000617  0.000201   

Ticker    VVA.

## Decomposing Vol

In [152]:
Omega_f_annual = Omega_f * 252
Psi_annual = Psi * 252

In [153]:
factor_var_daily = w.T @ factor_cov @ w
specific_var_daily = w.T @ Psi @ w
factor_var_annual = factor_var_daily * 252
specific_var_annual = specific_var_daily * 252

In [154]:
print("=== DAILY ===")
print("Total Vol:", portfolio_vol_daily)
print("Factor Vol:", np.sqrt(factor_var_daily))
print("Specific Vol:", np.sqrt(specific_var_daily))

print("\n=== ANNUAL ===")
print("Total Vol:", portfolio_vol_annual)
print("Factor Vol:", np.sqrt(factor_var_annual))
print("Specific Vol:", np.sqrt(specific_var_annual))

=== DAILY ===
Total Vol: 0.013049385278333657
Factor Vol: 0.0126330574364479
Specific Vol: 0.0068949535650829055

=== ANNUAL ===
Total Vol: 0.20715256925242945
Factor Vol: 0.20054356965141787
Specific Vol: 0.10945399460728543


In [155]:
portfolio_factor_exposure = w @ B_latest

factor_contributions_daily = (
    portfolio_factor_exposure *
    (Omega_f.values @ portfolio_factor_exposure)
)
factor_contributions_annual = factor_contributions_daily * 252

In [156]:
factor_contributions_df = pd.DataFrame({
    "Daily": factor_contributions_daily,
    "Annual": factor_contributions_annual
}, index=Omega_f.columns)

print(factor_contributions_df)

                    Daily        Annual
momentum     0.000000e+00  0.000000e+00
volatility  -2.043524e-20 -5.149679e-18
liquidity   -0.000000e+00 -0.000000e+00
size         0.000000e+00  0.000000e+00
Energy       2.852993e-05  7.189543e-03
Financials   2.240719e-05  5.646611e-03
Healthcare   1.814966e-05  4.573714e-03
Industrials  4.058403e-05  1.022717e-02
Materials    1.307527e-05  3.294967e-03


In [157]:
marginal_contrib = Sigma.values @ w
risk_contrib_daily = w * marginal_contrib
risk_contrib_annual = risk_contrib_daily * 252

risk_df = pd.DataFrame({
    "Daily": risk_contrib_daily,
    "Annual": risk_contrib_annual
}, index=returns.columns)

print(risk_df)

           Daily    Annual
Ticker                    
ANG.AX  0.000026  0.006657
AUB.AX  0.000014  0.003444
COS.AX  0.000025  0.006392
DUG.AX  0.000019  0.004803
LAU.AX  0.000010  0.002551
PLY.AX  0.000027  0.006799
TLX.AX  0.000017  0.004234
VVA.AX  0.000012  0.003137
WTC.AX  0.000005  0.001332
XYZ.AX  0.000014  0.003562
